# 3. KV Cache and FlashAttention
KV-cache is benchmarked directly on GPT-2. The FlashAttention section explains the correct experimental design and provides an optional same-model benchmark. The original notebook incorrectly compared GPT-2-large with TinyLlama.

In [ ]:
# KV-cache benchmark and optional FlashAttention-2 benchmark
%pip install -q -U transformers accelerate
import time, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
assert torch.cuda.is_available(), "Run this benchmark on a CUDA GPU."
device=torch.device("cuda")
MODEL="gpt2"
tok=AutoTokenizer.from_pretrained(MODEL); tok.pad_token=tok.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL,torch_dtype=torch.float16).to(device).eval()
prompt="Artificial intelligence is transforming the future of"
def bench(cache,runs=5):
    x=tok(prompt,return_tensors="pt").to(device)
    for _ in range(2):
        with torch.no_grad(): model.generate(**x,max_new_tokens=50,do_sample=False,use_cache=cache,pad_token_id=tok.eos_token_id)
    ts=[]
    for _ in range(runs):
        torch.cuda.synchronize(); t=time.perf_counter()
        with torch.no_grad(): model.generate(**x,max_new_tokens=50,do_sample=False,use_cache=cache,pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize(); ts.append(time.perf_counter()-t)
    return float(np.median(ts))
with_cache=bench(True); without_cache=bench(False)
print({"KV cache":with_cache,"no KV cache":without_cache,"speedup":without_cache/with_cache})

# FlashAttention must be compared using THE SAME model and weights. GPT-2 is not used for this section because the model architecture/backend support differs across versions.
# Example optional benchmark (requires a model that supports flash_attention_2 and the flash-attn package):
# FLASH_MODEL="Qwen/Qwen2.5-0.5B-Instruct"
# eager=AutoModelForCausalLM.from_pretrained(FLASH_MODEL,torch_dtype=torch.float16,attn_implementation="eager").to(device).eval()
# flash=AutoModelForCausalLM.from_pretrained(FLASH_MODEL,torch_dtype=torch.float16,attn_implementation="flash_attention_2").to(device).eval()
# Benchmark both with identical tokenizer, prompt, token count and warmup. Never compare two different model sizes and call the difference FlashAttention speedup.
